# Notebook 01: reference dataset and target identity

## Stage 1: structure, inputs, and computational provenance

This stage verifies the certified environment, reads target declarations from `config/project.yaml`, validates the external dataset manifest, and records the exact input state. It performs no sequence analysis, target identity resolution, candidate design, screening, or experimental planning.

In [ ]:
import zeaguard.repro as repro

repro.assert_environment_ready()

## Input contract

The notebook expects a versioned external reference dataset described by `data/reference/manifest.json`. Dataset files must be supplied locally under `data/external/` and listed with their project-relative paths and SHA-256 digests. The notebook never downloads or substitutes missing data. Unknown dataset name, version, origin, or acquisition timestamp remains `REVIEW`; absent files, unreadable files, malformed entries, and hash mismatches are critical `FAIL` conditions.

In [ ]:
from zeaguard import nb01_inputs

project_root = repro.find_project_root()
stage1 = nb01_inputs.prepare_nb01_stage1(project_root=project_root)

In [ ]:
declared_targets = stage1.targets
declared_targets

In [ ]:
print(stage1.report.to_text())
print(f"Precondition report: {stage1.report_path}")
print(f"Input provenance: {stage1.provenance_path}")

In [ ]:
stage1.report.raise_if_failed()

Stage 1 ends after the input preconditions pass. Sequence retrieval, target identity resolution, FASTA materialization, and scientific identity tables belong to a later stage.

## Stage 2: structured target identity anchors

This stage loads `data/reference/target_anchors.tsv` and validates its structure against the target declarations already loaded from `config/project.yaml`. It checks target coverage, unexpected targets, duplicate identifiers, required values, project roles, evidence classes, identity status, and explicitly unavailable published identifiers. No sequence retrieval or comparison occurs in this stage.

In [ ]:
stage2 = nb01_inputs.prepare_nb01_stage2(
    project_root=project_root,
    targets=declared_targets,
)

In [ ]:
stage2.anchors

In [ ]:
print(stage2.report.to_text())
print(f"Target anchor report: {stage2.report_path}")

In [ ]:
stage2.report.raise_if_failed()

Stage 2 ends with bibliographic anchors and their unresolved identifier gaps recorded. Computational identity resolution against the materialized reference dataset belongs to Stage 3.

## Stage 3: resolution of published identifiers against the public TSA

This stage reads the validated anchor table and manifest-selected TSA, then applies a fixed layered strategy: exact header identifier or alias matching, complete nucleotide query matching in forward and reverse-complement orientation, and local BLASTn only when exact evidence is absent. The visible thresholds and competitive-hit rules are defined in `zeaguard.nb01_identity` and persisted in the report. Outputs remain review candidates. No definitive target FASTA, candidate design, screening, or ranking is produced.

In [ ]:
from zeaguard import nb01_identity

stage3 = nb01_identity.prepare_nb01_stage3(project_root=project_root)

In [ ]:
stage3.decisions

In [ ]:
print(f"Identity candidates: {stage3.candidates_path}")
print(f"Identity report: {stage3.report_path}")

Stage 3 ends after recording reviewable identity candidates, metrics, ambiguities, and unresolved cases. Definitive transcript materialization requires scientific review of these decisions.